# 4 · Warm-start from the example model
Before starting, read [README.md](README.md). Run cells in order. Example files are from DECODE’s experiment, not the Nanoimager. Do not interpret a successful run as validated microscopy results. Nothing trains or writes data unless you explicitly enable it.
Fine-tuning starts with existing weights and continues training on new simulated data. It is not guaranteed to converge faster or transfer to your microscope; a suitable new PSF and noise model are still necessary. Here the example uses the **same example PSF and parameters**, solely to demonstrate the mechanism.

**Why the helper?** In the checked-out DECODE training code, `model_init` is present in the YAML but is not passed to the training model loader. Setting it alone can silently start from scratch. `finetune.py` calls the upstream engine, loads the state dict strictly into the model after trainer setup, and starts a fresh optimizer/scheduler. This is not checkpoint resumption. Different architecture or incompatible weights will produce an error. Verify the terminal prints `WARM START`. The helper does not modify DECODE or the original model. This path is based on the checked-out implementation and must be checked against the installed version on the lab PC.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tifffile

# Change this on the lab PC. It must contain experimental_data_workflow/.
ROOT = Path.cwd().resolve()
if not (ROOT / 'experimental_data_workflow').exists() and (ROOT.parent / 'experimental_data_workflow').exists():
    ROOT = ROOT.parent
EXAMPLE = ROOT / 'experimental_data_workflow'
WORK = ROOT / 'decode_work'
print('Project:', ROOT, '| Example exists:', EXAMPLE.exists())


In [ ]:
import decode.utils.param_io
import torch
import inspect
import decode.neuralfitter.train.train as trainer
print('Training module:', inspect.getfile(trainer))
print('setup_trainer signature:', inspect.signature(trainer.setup_trainer))
param = decode.utils.param_io.load_params(str(EXAMPLE / 'param_run.yaml'))
param.InOut.calibration_file = str((EXAMPLE / 'spline_calibration_3dcal.mat').resolve())
param.InOut.checkpoint_init = None
param.Hardware.device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
param.Hardware.device_simulation = param.Hardware.device
param.Hardware.num_worker_train = 0
param.HyperParameter.batch_size = 8
param.HyperParameter.epochs = 2
param.HyperParameter.pseudo_ds_size = 128
param.TestSet.test_size = 32
param.HyperParameter.opt_param.lr = 0.0001  # illustrative; requires tuning
print('Example weights:', EXAMPLE / 'model.pt')


In [ ]:
WRITE_CONFIG = False
if WRITE_CONFIG:
    WORK.mkdir(parents=True, exist_ok=True)
    (WORK / 'warmstart_runs').mkdir(exist_ok=True)
    param.InOut.experiment_out = str((WORK / 'warmstart_runs').resolve())
    config_path = WORK / 'warmstart_params.yaml'
    if config_path.exists(): raise FileExistsError(f'Choose a fresh path: {config_path}')
    decode.utils.param_io.save_params(str(config_path), param)
    helper = ROOT / 'decode_walkthrough' / 'finetune.py'
    print('In a terminal with the DECODE environment activated, run:')
    print(f'python "{helper}" --param "{config_path}" --weights "{EXAMPLE / "model.pt"}"')
else:
    print('Set WRITE_CONFIG = True to create the config and display the command.')


The helper creates its own new run directory via DECODE. `model.pt` is a weights file; DECODE’s `checkpoint_init` expects a training checkpoint with optimizer and scheduler state, so do not point it at `model.pt`. Evaluate any new model on independent examples and real data.
